In [1]:
from ingest import load_faq_data

documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)
        

153

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
import json

user_prompt = json.dumps(doc)

In [10]:
from evaluation_utils import llm_structured
from evaluation_utils import calc_price

In [11]:
result, answer_usage= llm_structured(
    openai_client, 
    data_gen_instructions,
    user_prompt,
    Questions
)

In [13]:
answer_usage
calc_price(answer_usage)

{'input_cost': 0.00015525, 'output_cost': 0.000459, 'total_cost': 0.00061425}

In [14]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course late — can I still sign up and join, or is it too late already?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to start the course now even if I missed the beginning?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for the certificate if I’m joining late?',
  'document': '74eb249bbf'},
 {'question': 'Are late joiners accepted, and is there a deadline for project submission?',
  'document': '74eb249bbf'}]

In [15]:
import pandas as pd

In [16]:
pd.DataFrame(records)

,question,document
0,I just found this course late — can I still si...,74eb249bbf
1,Am I allowed to start the course now even if I...,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,What do I need to do to qualify for the certif...,74eb249bbf
4,"Are late joiners accepted, and is there a dead...",74eb249bbf


In [17]:
from evaluation_utils import llm_structured_retry

In [21]:
def generate_ground_truth(doc):

    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "queston": q,
            "document": doc['id']
        })
    return results, usage
                         

In [19]:
generate_ground_truth(doc)

[{'queston': 'I just found this course, is it still okay to enroll and follow along?',
  'document': '74eb249bbf'},
 {'queston': 'Can latecomers still join the class, or is it too late now?',
  'document': '74eb249bbf'},
 {'queston': 'If I start the course after it’s already begun, can I still take part?',
  'document': '74eb249bbf'},
 {'queston': 'Am I allowed to join the course now, even though I discovered it late?',
  'document': '74eb249bbf'},
 {'queston': 'Do I need to worry about deadlines if I want the certificate and I’m joining after the course started?',
  'document': '74eb249bbf'}]

In [22]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [23]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [24]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/153 [00:00<?, ?it/s]

In [25]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

765

In [26]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.12404925000000006

In [27]:
from evaluation_utils import calc_total_price
calc_total_price(usages)

0.12404925000000006

In [28]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [30]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)